In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("future.no_silent_downcasting", True)

In [2]:
main_folder = "./../cleaned_data/EMA/"

df_behaviour = pd.read_csv(f"{main_folder}Behaviour_new.csv")
df_class = pd.read_csv(f"{main_folder}Class_new.csv")
df_events = pd.read_csv(f"{main_folder}Events.csv")
df_exercise = pd.read_csv(f"{main_folder}Exercise.csv")
df_mood = pd.read_csv(f"{main_folder}Mood_new.csv")
df_sleep = pd.read_csv(f"{main_folder}Sleep.csv")
df_social = pd.read_csv(f"{main_folder}Social.csv")
df_stress = pd.read_csv(f"{main_folder}Stress_new.csv")
df_class2 = pd.read_csv(f"{main_folder}Class2.csv")

In [3]:
col_behaviour = [
    "user_id",
    "datetime_utc",
    "anxious",
    "calm",
    "conventional",
    "critical",
    "dependable",
    "disorganized",
    "enthusiastic",
    "experiences",
    "reserved",
    "sympathetic",
    # "resp_time",
]
df_behaviour = df_behaviour[col_behaviour]

col_class = [
    "user_id",
    "datetime_utc",
    "due",
    "experience",
    "hours",
    # "course_id", --> Removing course_id and assuming same courses for all in this df
    # "location",
    # "resp_time",
]
df_class["course_id"] = (
    df_class["course_id"].astype(str).str.strip().str.lower().replace("nan", pd.NA)
)
df_class = df_class[col_class]


col_class2 = [
    "user_id",
    "datetime_utc",
    "challenge",
    "effort",
    "grade",
    # "resp_time",
    # "location",
]

df_class2 = df_class2[col_class2]


col_events = [
    "user_id",
    "datetime_utc",
    "positive_event_score",
    "negative_event_score",
    "emotion_range",
    "has_positive_text",
    "has_negative_text",
    # "positive_text",
    # "negative_text",
    # "resp_time",
    # "location",
]
df_events = df_events[col_events]


df_exercise = df_exercise.rename(columns={"resp_time": "datetime_utc"})
col_exercise = [
    "user_id",
    "datetime_utc",
    "exercise",
    "walk",
    "have",
    "schedule",
    # "weekday",
    # "hour",
    # "latitude",
    # "longitude",
    # "has_location",
]
df_exercise = df_exercise[col_exercise]


col_mood = [
    "user_id",
    "datetime_utc",
    "happy",
    "happyornot",
    "sad",
    "sadornot",
    # "location",
    # "resp_time",
]
df_mood = df_mood[col_mood]


col_sleep = [
    "user_id",
    "datetime_utc",
    "sleep_hours",
    "sleep_quality",
    "sleepiness",
    # "resp_time",
    # "location",
]
df_sleep = df_sleep[col_sleep]


col_social = [
    "user_id",
    "datetime_utc",
    "number",
    # "resp_time",
    # "location"
]
df_social = df_social[col_social]


col_stress = [
    "user_id",
    "datetime_utc",
    "level",
    # "location",
    # "resp_time"
]
df_stress = df_stress[col_stress]

In [4]:
def fill_daily(df, user_col="user_id", time_col="datetime_utc"):
    df = df.copy()
    df["date"] = pd.to_datetime(df[time_col]).dt.floor("D")

    # Get behavior columns (everything except IDs and time)
    value_cols = [col for col in df.columns if col not in [user_col, time_col, "date"]]

    # Daily aggregation
    df_daily = df.groupby([user_col, "date"])[value_cols].mean().reset_index()

    filled_rows = []

    for user_id, group in df_daily.groupby(user_col):
        group = group.set_index("date").sort_index()
        full_range = pd.date_range(
            start=group.index.min(), end=group.index.max(), freq="D"
        )

        expanded = pd.DataFrame(index=full_range, columns=value_cols)
        expanded.update(group)

        expanded = expanded.ffill().bfill()
        expanded = expanded.infer_objects(copy=False)
        expanded[user_col] = user_id
        expanded = expanded.reset_index().rename(columns={"index": "date"})

        filled_rows.append(expanded)

    df_filled = pd.concat(filled_rows, ignore_index=True)
    df_filled = df_filled.sort_values([user_col, "date"]).reset_index(drop=True)

    return df_filled

## 1. Behaviour

In [5]:
filled_behaviour_daily = fill_daily(df_behaviour)

# Chack for na
filled_behaviour_daily_na = filled_behaviour_daily[
    filled_behaviour_daily.isna().any(axis=1)
]
# print(filled_behaviour_daily)

In [6]:
# Define columns and Likert scale
behavior_columns = [
    "anxious",
    "calm",
    "conventional",
    "critical",
    "dependable",
    "disorganized",
    "enthusiastic",
    "experiences",
    "reserved",
    "sympathetic",
]
likert_5 = ["1", "2", "3", "4", "5"]

for col in behavior_columns:
    if col in filled_behaviour_daily.columns:
        # Step 1: Fill NaNs with neutral (3.0)
        filled_behaviour_daily[col] = filled_behaviour_daily[col].fillna(3.0)

        # Step 2: Round to nearest Likert point
        filled_behaviour_daily[col] = filled_behaviour_daily[col].round().astype(int)

        # Step 3: Convert to string (for mapping to categories)
        filled_behaviour_daily[col] = filled_behaviour_daily[col].astype(str)

        # Step 4: Force invalid entries to neutral
        filled_behaviour_daily[col] = filled_behaviour_daily[col].apply(
            lambda x: x if x in likert_5 else "3"
        )

        # Step 5: Convert to ordered categorical
        filled_behaviour_daily[col] = pd.Categorical(
            filled_behaviour_daily[col], categories=likert_5, ordered=True
        ).codes  # Output: 0–4
# 0 = "1", 2 = "3", 4 = "5"

print(filled_behaviour_daily.dtypes)
print(filled_behaviour_daily.head(10))

date            datetime64[ns, UTC]
anxious                        int8
calm                           int8
conventional                   int8
critical                       int8
dependable                     int8
disorganized                   int8
enthusiastic                   int8
experiences                    int8
reserved                       int8
sympathetic                    int8
user_id                      object
dtype: object
                       date  anxious  calm  conventional  critical  \
0 2013-04-08 00:00:00+00:00        3     3             4         1   
1 2013-04-09 00:00:00+00:00        3     3             4         1   
2 2013-04-10 00:00:00+00:00        4     2             2         0   
3 2013-04-11 00:00:00+00:00        4     2             2         0   
4 2013-04-12 00:00:00+00:00        4     2             2         0   
5 2013-04-13 00:00:00+00:00        4     2             2         0   
6 2013-04-14 00:00:00+00:00        4     2             2        

## 2. class

In [7]:
filled_class_daily = fill_daily(df_class)

In [8]:
# Define ordinal categories for experience and hours
ordinal_map_class = {
    "experience": [
        "strongly disagree",
        "disagree",
        "neutral",
        "agree",
        "strongly agree",
    ],
    "hours": ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", ">10"],
}

# Fill experience and hours by mapping float → index → label → ordinal code
for col, categories in ordinal_map_class.items():
    if col in filled_class_daily.columns:
        # Step 1: Choose a neutral fallback index if missing
        midpoint = len(categories) // 2
        filled_class_daily[col] = filled_class_daily[col].fillna(midpoint)

        # Step 2: Round float averages to nearest index
        filled_class_daily[col] = filled_class_daily[col].round().astype(int)

        # Step 3: Map index to category string (e.g., 3 → "agree")
        filled_class_daily[col] = filled_class_daily[col].apply(
            lambda x: (
                categories[x] if 0 <= x < len(categories) else categories[midpoint]
            )
        )

        # Step 4: Convert to ordered categorical and then to numeric ordinal codes
        filled_class_daily[col] = pd.Categorical(
            filled_class_daily[col], categories=categories, ordered=True
        ).codes

# print(df_class_filled["due"].value_counts(dropna=False))
# Converting averages to binary (0 = No, 1 = Yes)
filled_class_daily["due"] = filled_class_daily["due"].apply(
    lambda x: 1 if pd.notna(x) and x <= 1.5 else 0
)

print(filled_class_daily.dtypes)
print(filled_class_daily.head(10))

date          datetime64[ns, UTC]
due                         int64
experience                   int8
hours                        int8
user_id                    object
dtype: object
                       date  due  experience  hours user_id
0 2013-03-26 00:00:00+00:00    0           3      1     u00
1 2013-03-27 00:00:00+00:00    1           1      1     u00
2 2013-03-28 00:00:00+00:00    0           2      2     u00
3 2013-03-29 00:00:00+00:00    0           3      2     u00
4 2013-03-30 00:00:00+00:00    0           3      3     u00
5 2013-03-31 00:00:00+00:00    0           3      3     u00
6 2013-04-01 00:00:00+00:00    1           2      2     u00
7 2013-04-02 00:00:00+00:00    0           3      3     u00
8 2013-04-03 00:00:00+00:00    0           3      3     u00
9 2013-04-04 00:00:00+00:00    0           3      7     u00


## 3. Class 2

In [9]:
filled_class2_daily = fill_daily(df_class2)
print(filled_class2_daily.head())

                       date  challenge  effort  grade user_id
0 2013-04-25 00:00:00+00:00        1.0     1.0    4.0     u00
1 2013-04-26 00:00:00+00:00        1.0     1.0    4.0     u00
2 2013-04-27 00:00:00+00:00        1.0     1.0    4.0     u00
3 2013-04-28 00:00:00+00:00        1.0     1.0    4.0     u00
4 2013-04-29 00:00:00+00:00        1.0     1.0    4.0     u00


In [10]:
# Define the ordinal categories in order (from strongest agreement to strongest disagreement, or highest to lowest grade)
ordinal_categories = {
    "challenge": [
        "Agree Strongly",
        "Agree Moderately",
        "Agree Slightly",
        "Disagree Slightly",
        "Disagree Moderately",
        "Disagree Strongly",
    ],
    "effort": [
        "Agree Strongly",
        "Agree Moderately",
        "Agree Slightly",
        "Disagree Slightly",
        "Disagree Moderately",
        "Disagree Strongly",
    ],
    "grade": ["A", "A-", "B+", "B", "B-", "C+", "C", "C-"],
}

# Loop through each column to apply ordinal conversion
for col, categories in ordinal_categories.items():
    if col in filled_class2_daily.columns:
        # Step 1: Fill missing values with the middle (neutral) index of the scale
        midpoint = len(categories) // 2
        filled_class2_daily[col] = filled_class2_daily[col].fillna(midpoint)

        # Step 2: Round the float to the nearest integer (corresponds to category index)
        filled_class2_daily[col] = filled_class2_daily[col].round().astype(int)

        # Step 3: Map the rounded index to its corresponding string label
        filled_class2_daily[col] = filled_class2_daily[col].apply(
            lambda x: (
                categories[x] if 0 <= x < len(categories) else categories[midpoint]
            )
        )

        # Step 4: Convert to ordered categorical and then to numeric code (0 = lowest)
        filled_class2_daily[col] = pd.Categorical(
            filled_class2_daily[col], categories=categories, ordered=True
        ).codes

# print(filled_class2_daily.dtypes)
# print(filled_class2_daily.head(10))

## 4. Events

In [11]:
filled_events_daily = fill_daily(df_events)
print(filled_events_daily.head())

                       date  positive_event_score  negative_event_score  \
0 2013-04-18 00:00:00+00:00                   5.0                   5.0   
1 2013-04-19 00:00:00+00:00                   5.0                   5.0   
2 2013-04-20 00:00:00+00:00                   5.0                   5.0   
3 2013-04-21 00:00:00+00:00                   5.0                   5.0   
4 2013-04-22 00:00:00+00:00                   5.0                   5.0   

   emotion_range  has_positive_text  has_negative_text user_id  
0            0.0                1.0                1.0     u00  
1            0.0                1.0                1.0     u00  
2            0.0                1.0                1.0     u00  
3            0.0                1.0                1.0     u00  
4            0.0                1.0                1.0     u00  


In [12]:
# These two columns are Likert-style ratings from 1 (very weak) to 7 (very intense)
ordinal_7 = ["positive_event_score", "negative_event_score"]

for col in ordinal_7:
    if col in filled_events_daily.columns:
        filled_events_daily[col] = (
            filled_events_daily[col]
            .fillna(4)  # Use 4 as a neutral fallback
            .astype(float)
            .round()
            .clip(lower=1, upper=7)  # Clamp to valid range
            .astype(int)
            - 1  # Convert to ordinal code: 1–7 → 0–6
        )

# These indicate whether the user provided a text description or not
binary_cols = ["has_positive_text", "has_negative_text"]

for col in binary_cols:
    if col in filled_events_daily.columns:
        filled_events_daily[col] = (
            filled_events_daily[col]
            .fillna(0)  # Treat missing as "No"
            .astype(float)
            .round()
            .astype(int)
            .clip(0, 1)  # Ensure values are strictly binary
        )

# This is a continuous value indicating the emotional gap between events
if "emotion_range" in filled_events_daily.columns:
    filled_events_daily["emotion_range"] = filled_events_daily["emotion_range"].fillna(
        0.0
    )

# print(filled_events_daily.head(25))
# print(filled_events_daily.dtypes)

## 5. Exercise

In [13]:
# Step 1: Fill daily data
filled_exercise_daily = fill_daily(df_exercise)

In [14]:
# These use a 5-point ordinal scale for time spent exercising or walking
ordinal_exercise_cols = ["exercise", "walk"]
likert_5 = ["1", "2", "3", "4", "5"]

for col in ordinal_exercise_cols:
    if col in filled_exercise_daily.columns:
        filled_exercise_daily[col] = filled_exercise_daily[col].fillna(3.0)

        filled_exercise_daily[col] = (
            filled_exercise_daily[col]
            .astype(float)
            .round()
            .clip(1, 5)
            .astype(int)
            .astype(str)
        )

        filled_exercise_daily[col] = filled_exercise_daily[col].apply(
            lambda x: x if x in likert_5 else "3"
        )

        filled_exercise_daily[col] = pd.Categorical(
            filled_exercise_daily[col], categories=likert_5, ordered=True
        ).codes

# These are Yes/No binary responses encoded as 1/2 → Yes, everything else → No
binary_exercise_cols = ["have", "schedule"]

for col in binary_exercise_cols:
    if col in filled_exercise_daily.columns:
        filled_exercise_daily[col] = filled_exercise_daily[col].apply(
            lambda x: 1 if pd.notna(x) and float(x) <= 2 else 0
        )

print(filled_exercise_daily.head())
print(filled_exercise_daily.dtypes)

                       date  exercise  walk  have  schedule user_id
0 2013-04-01 00:00:00+00:00         1     1     1         1     u00
1 2013-04-02 00:00:00+00:00         0     0     1         1     u00
2 2013-04-03 00:00:00+00:00         3     1     1         1     u00
3 2013-04-04 00:00:00+00:00         0     1     1         1     u00
4 2013-04-05 00:00:00+00:00         3     2     1         1     u00
date        datetime64[ns, UTC]
exercise                   int8
walk                       int8
have                      int64
schedule                  int64
user_id                  object
dtype: object


## 6. mood

In [15]:
# Step 1: Fill daily values using your main function
filled_mood_daily = fill_daily(df_mood)

In [16]:
# These use a 4-point ordinal scale (1 = a little bit, 4 = extremely)
ordinal_mood_cols = ["happy", "sad"]
likert_4 = ["1", "2", "3", "4"]

for col in ordinal_mood_cols:
    if col in filled_mood_daily.columns:
        filled_mood_daily[col] = filled_mood_daily[col].fillna(
            2.0
        )  # midpoint = "somewhat"

        filled_mood_daily[col] = (
            filled_mood_daily[col]
            .astype(float)
            .round()
            .clip(1, 4)
            .astype(int)
            .astype(str)
        )

        filled_mood_daily[col] = filled_mood_daily[col].apply(
            lambda x: x if x in likert_4 else "2"
        )

        filled_mood_daily[col] = pd.Categorical(
            filled_mood_daily[col], categories=likert_4, ordered=True
        ).codes  # ordinal: 0–3

binary_mood_cols = ["happyornot", "sadornot"]

for col in binary_mood_cols:
    if col in filled_mood_daily.columns:
        filled_mood_daily[col] = filled_mood_daily[col].apply(
            lambda x: 1 if pd.notna(x) and float(x) <= 2 else 0
        )

print(filled_mood_daily.head())
print(filled_mood_daily.dtypes)

                       date  happy  happyornot  sad  sadornot user_id
0 2013-04-25 00:00:00+00:00      1           1    3         1     u00
1 2013-04-26 00:00:00+00:00      1           1    3         1     u00
2 2013-04-27 00:00:00+00:00      1           1    3         1     u00
3 2013-04-28 00:00:00+00:00      1           1    3         1     u00
4 2013-04-29 00:00:00+00:00      1           1    3         1     u00
date          datetime64[ns, UTC]
happy                        int8
happyornot                  int64
sad                          int8
sadornot                    int64
user_id                    object
dtype: object


## 7. sleep

In [17]:
# Step 1: Fill daily values
filled_sleep_daily = fill_daily(df_sleep)

In [18]:
ordinal_4_sleep = {
    "sleep_quality": ["1", "2", "3", "4"],  # Very good → Very bad
    "sleepiness": ["1", "2", "3", "4"],  # None → Three or more times
}

for col, categories in ordinal_4_sleep.items():
    if col in filled_sleep_daily.columns:
        midpoint = len(categories) // 2  # Neutral fallback: index 2
        filled_sleep_daily[col] = filled_sleep_daily[col].fillna(midpoint)

        filled_sleep_daily[col] = (
            filled_sleep_daily[col]
            .astype(float)
            .round()
            .clip(1, 4)
            .astype(int)
            .astype(str)
        )

        filled_sleep_daily[col] = filled_sleep_daily[col].apply(
            lambda x: x if x in categories else categories[midpoint]
        )

        filled_sleep_daily[col] = pd.Categorical(
            filled_sleep_daily[col], categories=categories, ordered=True
        ).codes  # 0–3


# Treat as ordinal: [1] = <3 hrs, [19] = 12 hrs
if "sleep_hours" in filled_sleep_daily.columns:
    filled_sleep_daily["sleep_hours"] = (
        filled_sleep_daily["sleep_hours"]
        .fillna(10.0)  # fallback ~ "8 hours" → index 11
        .astype(float)
        .round()
        .clip(1, 19)
        .astype(int)
        - 1  # convert to 0-based ordinal codes (0–18)
    )

print(filled_sleep_daily.head())
print(filled_sleep_daily.dtypes)

                       date  sleep_hours  sleep_quality  sleepiness user_id
0 2013-03-24 00:00:00+00:00            7              0           0     u00
1 2013-03-25 00:00:00+00:00            7              0           0     u00
2 2013-03-26 00:00:00+00:00            7              1           0     u00
3 2013-03-27 00:00:00+00:00            7              1           0     u00
4 2013-03-28 00:00:00+00:00            5              1           0     u00
date             datetime64[ns, UTC]
sleep_hours                    int64
sleep_quality                   int8
sleepiness                      int8
user_id                       object
dtype: object


## 8. social

In [19]:
# Step 1: Fill daily values
filled_social_daily = fill_daily(df_social)

In [20]:
# 'number' is an ordinal scale based on how many people the participant interacted with:
# [1] 0–4, [2] 5–9, [3] 10–19, [4] 20–49, [5] 50–99, [6] 100+
if "number" in filled_social_daily.columns:
    # Step 1: Fill missing values with the midpoint (3.0 → "10–19 people")
    filled_social_daily["number"] = filled_social_daily["number"].fillna(3.0)

    # Step 2: Round to the nearest valid category, clip to stay within 1–6
    filled_social_daily["number"] = (
        filled_social_daily["number"]
        .astype(float)
        .round()
        .clip(1, 6)
        .astype(int)
        .astype(str)
    )

    # Step 3: Clean any outliers (e.g., weird string values) by defaulting to neutral category "3"
    filled_social_daily["number"] = filled_social_daily["number"].apply(
        lambda x: x if x in ["1", "2", "3", "4", "5", "6"] else "3"
    )

    # Step 4: Convert to ordered categorical variable and extract ordinal code (0 = smallest group, 5 = largest)
    filled_social_daily["number"] = pd.Categorical(
        filled_social_daily["number"],
        categories=["1", "2", "3", "4", "5", "6"],
        ordered=True,
    ).codes

print(filled_social_daily.head())
print(filled_social_daily.dtypes)

                       date  number user_id
0 2013-03-24 00:00:00+00:00       1     u00
1 2013-03-25 00:00:00+00:00       1     u00
2 2013-03-26 00:00:00+00:00       1     u00
3 2013-03-27 00:00:00+00:00       3     u00
4 2013-03-28 00:00:00+00:00       2     u00
date       datetime64[ns, UTC]
number                    int8
user_id                 object
dtype: object


## 9. Stress

In [21]:
# Step 1: Fill daily values using your main function
filled_stress_daily = fill_daily(df_stress)

In [22]:
# 'level' is an ordinal scale where lower values indicate more stress and higher values mean feeling good/great
if "level" in filled_stress_daily.columns:
    # Step 1: Fill missing values with a neutral fallback ("Stressed out" = 3.0)
    filled_stress_daily["level"] = filled_stress_daily["level"].fillna(3.0)

    # Step 2: Round and clip values to keep them in the valid 1–5 range
    filled_stress_daily["level"] = (
        filled_stress_daily["level"]
        .astype(float)
        .round()
        .clip(1, 5)
        .astype(int)
        .astype(str)
    )

    # Step 3: Handle any unexpected values by defaulting to the neutral option ("3")
    filled_stress_daily["level"] = filled_stress_daily["level"].apply(
        lambda x: x if x in ["1", "2", "3", "4", "5"] else "3"
    )

    # Step 4: Convert to an ordered categorical and extract ordinal codes (0 = most stressed, 4 = feeling great)
    filled_stress_daily["level"] = pd.Categorical(
        filled_stress_daily["level"], categories=["1", "2", "3", "4", "5"], ordered=True
    ).codes

print(filled_stress_daily.head())
print(filled_stress_daily.dtypes)

                       date  level user_id
0 2013-03-25 00:00:00+00:00      1     u00
1 2013-03-26 00:00:00+00:00      1     u00
2 2013-03-27 00:00:00+00:00      0     u00
3 2013-03-28 00:00:00+00:00      2     u00
4 2013-03-29 00:00:00+00:00      1     u00
date       datetime64[ns, UTC]
level                     int8
user_id                 object
dtype: object


# Merging all daily datasets

In [23]:
from functools import reduce

# Step 1: List of all cleaned daily DataFrames
filled_dfs_list = [
    filled_behaviour_daily,
    filled_class_daily,
    filled_class2_daily,
    filled_events_daily,
    filled_exercise_daily,
    filled_mood_daily,
    filled_sleep_daily,
    filled_social_daily,
    filled_stress_daily,
]

# Step 2: Merge all DataFrames on ['user_id', 'date'] using outer join
merged_df = reduce(
    lambda left, right: pd.merge(left, right, on=["user_id", "date"], how="outer"),
    filled_dfs_list,
)

# Step 3: Sort by user and date for time series modeling
merged_df = merged_df.sort_values(by=["user_id", "date"]).reset_index(drop=True)

print("Merged DataFrame shape:", merged_df.shape)
print("Columns:", merged_df.columns.tolist())
display(merged_df.head(100))

Merged DataFrame shape: (2883, 36)
Columns: ['date', 'anxious', 'calm', 'conventional', 'critical', 'dependable', 'disorganized', 'enthusiastic', 'experiences', 'reserved', 'sympathetic', 'user_id', 'due', 'experience', 'hours', 'challenge', 'effort', 'grade', 'positive_event_score', 'negative_event_score', 'emotion_range', 'has_positive_text', 'has_negative_text', 'exercise', 'walk', 'have', 'schedule', 'happy', 'happyornot', 'sad', 'sadornot', 'sleep_hours', 'sleep_quality', 'sleepiness', 'number', 'level']


,date,anxious,calm,conventional,critical,dependable,disorganized,enthusiastic,experiences,reserved,sympathetic,user_id,due,experience,hours,challenge,effort,grade,positive_event_score,negative_event_score,emotion_range,has_positive_text,has_negative_text,exercise,walk,have,schedule,happy,happyornot,sad,sadornot,sleep_hours,sleep_quality,sleepiness,number,level
0,2013-03-24 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,0.0,0.0,1.0,NaN
1,2013-03-25 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,0.0,0.0,1.0,1.0
2,2013-03-26 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,0.0,3.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,1.0,0.0,1.0,1.0
3,2013-03-27 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.0,1.0,0.0,3.0,0.0
4,2013-03-28 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,0.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0,0.0,2.0,2.0
5,2013-03-29 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,0.0,3.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0,0.0,3.0,1.0
6,2013-03-30 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,0.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0,0.0,3.0,3.0
7,2013-03-31 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,0.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.0,1.0,0.0,1.0,3.0
8,2013-04-01 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,1.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,2.0,3.0,0.0,1.0,3.0
9,2013-04-02 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,u00,0.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,2.0,3.0,0.0,1.0,0.0


In [24]:
# Forward and backward fill within each user's timeline
merged_df_filled = merged_df.groupby("user_id").ffill().bfill()

display(merged_df_filled.head(100))

,date,anxious,calm,conventional,critical,dependable,disorganized,enthusiastic,experiences,reserved,sympathetic,due,experience,hours,challenge,effort,grade,positive_event_score,negative_event_score,emotion_range,has_positive_text,has_negative_text,exercise,walk,have,schedule,happy,happyornot,sad,sadornot,sleep_hours,sleep_quality,sleepiness,number,level
0,2013-03-24 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,0.0,3.0,1.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,7.0,0.0,0.0,1.0,1.0
1,2013-03-25 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,0.0,3.0,1.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,7.0,0.0,0.0,1.0,1.0
2,2013-03-26 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,0.0,3.0,1.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,7.0,1.0,0.0,1.0,1.0
3,2013-03-27 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,1.0,1.0,1.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,7.0,1.0,0.0,3.0,0.0
4,2013-03-28 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,0.0,2.0,2.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,5.0,1.0,0.0,2.0,2.0
5,2013-03-29 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,0.0,3.0,2.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,5.0,1.0,0.0,3.0,1.0
6,2013-03-30 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,0.0,3.0,3.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,5.0,1.0,0.0,3.0,3.0
7,2013-03-31 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,0.0,3.0,3.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,8.0,1.0,0.0,1.0,3.0
8,2013-04-01 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,1.0,2.0,2.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,2.0,3.0,0.0,1.0,3.0
9,2013-04-02 00:00:00+00:00,3.0,3.0,4.0,1.0,2.0,2.0,3.0,3.0,2.0,3.0,0.0,3.0,3.0,1.0,1.0,4.0,4.0,4.0,0.000000,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,3.0,1.0,2.0,3.0,0.0,1.0,0.0


In [25]:
merged_df.to_csv("./lstm_process_data/cleaned_daily_data.csv", index=False)